![databricks_academy_logo.png](../Includes/images/databricks_academy_logo.png "databricks_academy_logo.png")

# Task 2 - Silver - Gold Table
This notebook is used for task 2 in the job from the directions in notebook: **Jobs - Creating a Simple Lakeflow Job**

## Capture Job Parameters

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")

## Configure Your Environment

1. Set the default catalog and schema.

In [0]:
# Set the catalog and schema
spark.sql(f'USE CATALOG {catalog_name}')
spark.sql(f'USE SCHEMA {schema_name}')

### SILVER
**Objective**: Transform the bronze table and create the silver table.

1. Create a table named **current_employees_silver_job** from the **current_employees_bronze_job** table. 

    The table will:
    - Select the columns **ID**, **FirstName**, **Country**.
    - Convert the **Role** column to uppercase.
    - Add two new columns: **CurrentTimeStamp** and **CurrentDate**.

In [0]:
%sql
-- Create a temporary view to use to merge the data into the final silver table
CREATE OR REPLACE TABLE current_employees_silver_job AS 
SELECT 
  ID,
  FirstName,
  Country,
  upper(Role) as Role,                 -- Upcase the Role column
  current_timestamp() as CurrentTimeStamp,    -- Get the current datetime
  date(CurrentTimeStamp) as CurrentDate       -- Get the date
FROM current_employees_bronze_job;

### GOLD
**Objective:** Aggregate the silver table to create the final gold table.

1. Create a temporary view named **temp_view_total_roles_job** that aggregates the total number of employees by role. Then, display the results of the view.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW temp_view_total_roles_job AS 
SELECT
  Role, 
  count(*) as TotalEmployees
FROM current_employees_silver_job
GROUP BY Role;

2. Create the final gold table named **total_roles_gold_job** with the specified columns.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS total_roles_gold_job (
  Role STRING,
  TotalEmployees INT
);

3. Insert all rows from the aggregated temporary view **temp_view_total_roles_job** into the **total_roles_gold_job** table, overwriting the existing data in the table. This overwrites the data in a table but keeps the existing schema and table definition and properties.

    Confirm the following:
    - **num_affected_rows** is *4*
    - **num_inserted_rows** is *4*

In [0]:
%sql
INSERT OVERWRITE TABLE total_roles_gold_job
SELECT * 
FROM temp_view_total_roles_job;